In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install rdkit -q
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 65.0 MB/s eta 0:00:00
CUDA: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/ali1810/Water_Solubility
%cd Water_Solubility
!ls

Cloning into 'Water_Solubility'...
remote: Enumerating objects: 3288, done.
remote: Counting objects: 100% (1302/1302), done.
remote: Compressing objects: 100% (1166/1166), done.
remote: Total 3288 (delta 171), reused 1221 (delta 127), pack-reused 1986 (from 3)
Receiving objects: 100% (3288/3288), 34.33 MiB | 24.62 MiB/s, done.
Resolving deltas: 100% (492/492), done.
Filtering content: 100% (9/9), 69.49 MiB | 11.12 MiB/s, done.
/content/Water_Solubility
Dockerfile			  README.md
env				  requirements3.txt
feature_order.json		  requirements.txt
final_unique_test.csv		  runtime.txt
final_unique_train.csv		  training_set_vectors.npy
footer.py			  train_mpnn.py
model_evaluation_test_data.ipynb  utilities.py
mpnn_model.py			  Water-Solubility1.py
pages				  Water-Solubility.py
Procfile			  wf.png
__pycache__			  xgboost_model_298_4045.json


In [ ]:
import os
needed = ['final_unique_train.csv', 'final_unique_test.csv',
          'mpnn_model.py', 'train_mpnn.py']
for f in needed:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"{status}  {f}")

✅  final_unique_train.csv
✅  final_unique_test.csv
✅  mpnn_model.py
✅  train_mpnn.py


In [ ]:
import pandas as pd
df = pd.read_csv('final_unique_train.csv')
print("Columns:", list(df.columns))
print("Rows:", len(df))
print(df.head(3))

Columns: ['Unnamed: 0', 'C_ID', 'Name', 'InChIKey', 'SMILES', 'CAS', 'LogS', 'smiles_canon']
Rows: 17937
   Unnamed: 0  C_ID Name                     InChIKey  \
0           0  A-51  NaN  BOCJQSFSGAZAPQ-UHFFFAOYSA-N   
1           1  A-53  NaN  BRBKOPJOKNSWSG-UHFFFAOYSA-N   
2           2  A-55  NaN  BRPOVUNVTUCYNN-UHFFFAOYSA-N   

                                              SMILES  CAS     LogS  \
0                   Clc1c2C(=O)c3c(C(=O)c2ccc1)cccc3  NaN -5.54000   
1                      S(=O)(=O)(N=C(N)N)c1ccc(N)cc1  NaN -1.98497   
2  Fc1c(N2C(CC)C(N)C2)cc2n(C3CC3)cc(C(=O)O)c(=O)c2c1  NaN -3.91200   

                                    smiles_canon  
0                 O=C1c2ccccc2C(=O)c2c(Cl)cccc21  
1                    NC(N)=NS(=O)(=O)c1ccc(N)cc1  
2  CCC1C(N)CN1c1cc2c(cc1F)c(=O)c(C(=O)O)cn2C1CC1  


In [ ]:
!sed -i 's/ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)/ReduceLROnPlateau(optimizer, patience=5, factor=0.5)/' train_mpnn.py
print("Fixed")

Fixed


In [ ]:
!sed -i 's/ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)/ReduceLROnPlateau(optimizer, patience=5, factor=0.5)/' mpnn_model.py
print("Fixed")

Fixed


In [ ]:
!grep "ReduceLROnPlateau" mpnn_model.py


  Scheduler: ReduceLROnPlateau
from torch.optim.lr_scheduler import ReduceLROnPlateau
    scheduler = ReduceLROnPlateau(optimizer, patience=5, factor=0.5)


In [ ]:
!sed -i 's/node_in:    int = ATOM_DIM,    # 51/node_in:    int = 54,         # 54/' mpnn_model.py
print("Fixed")

Fixed


In [ ]:
!python train_mpnn.py \
    --train_csv final_unique_train.csv \
    --test_csv  final_unique_test.csv \
    --output_dir models \
    --epochs 100 \
    --device cuda

MPNN Solubility Model — Training
Dr. Mushtaq Ali · KIT
Train CSV: final_unique_train.csv
Test CSV:  final_unique_test.csv
Output:    models

Training on: cuda
Dataset: 17937 rows | target='LogS'
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] Explicit valence for atom # 5 N, 4, is greater than permitted
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] Explicit valence for atom # 5 N, 4, is greater than permitted
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56] WARNING: not removing hydrogen atom without neighbors
[13:49:56

In [ ]:
import os
print("Best model exists:", os.path.exists('models/mpnn_solubility_best.pt'))
print("Size:", os.path.getsize('models/mpnn_solubility_best.pt'), "bytes")

Best model exists: True
Size: 3634513 bytes


In [ ]:
from mpnn_model import MPNNSolubility

model = MPNNSolubility.load('models/mpnn_solubility_best.pt')

molecules = [
    ('CCO',                          'Ethanol',   '+0.9'),
    ('c1ccccc1',                     'Benzene',   '-1.6'),
    ('CC(=O)OC1=CC=CC=C1C(=O)O',   'Aspirin',   '-2.1'),
    ('Cn1cnc2c1c(=O)n(C)c(=O)n2C', 'Caffeine',  '-1.2'),
]

print(f"{'Molecule':<15} {'MPNN':>8}  {'Expected':>10}")
print("-" * 38)
for smi, name, exp in molecules:
    logS = model.predict_smiles(smi)
    print(f"{name:<15} {logS:>8.3f}  {exp:>10}")

Molecule            MPNN    Expected
--------------------------------------
Ethanol            1.282        +0.9
Benzene           -2.015        -1.6
Aspirin           -1.436        -2.1
Caffeine          -1.978        -1.2


In [ ]:
from google.colab import files
files.download('models/mpnn_solubility_best.pt')
print("Done — model saved to your computer")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done — model saved to your computer
